# Fuzzy learning-degree comparison

Generated thin entry point. All method logic lives in `src/`.

In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

REPO = Path(os.environ["NEURON_DEATH_SOURCE"])
DATA = Path(os.environ["NEURON_DEATH_DATA"])
assert (REPO / "src/train.py").is_file()
assert (DATA / "mnist.npz").is_file()
os.chdir(REPO)
print("CUDA devices:", __import__("torch").cuda.device_count(), flush=True)
assert __import__("torch").cuda.device_count() == 2, "This job requires two independent T4 GPUs"
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q",
                "-p", "no:cacheprovider"], cwd=REPO, check=True)

configs = sorted(glob.glob(str(REPO / "configs/fuzzy_v1/*.json")))
assert len(configs) == 35, f"expected 35 frozen configs, found {len(configs)}"
plan = __import__("json").loads((REPO / "configs/fuzzy_v1_plan.json").read_text())
assert plan["status"] == "FROZEN_BEFORE_FIRST_RUN"
assert plan["setting"]["seeds"] == [10, 11, 12, 13, 14]
RUNS = Path("/kaggle/working/fuzzy_v1_runs")
print("FUZZY_MAIN_LAUNCHING", flush=True)
subprocess.run([sys.executable, "scripts/launch_pair.py", *configs,
                "--gpus", "0,1", "--runs-root", str(RUNS),
                "--data-root", str(DATA), "--budget-hours", "10.5"],
               cwd=REPO, check=True)
EXTRACT = Path("/kaggle/working/fuzzy_v1_extract")
subprocess.run([sys.executable, "scripts/make_analysis_extract.py",
                "--runs-root", str(RUNS), "--pattern", "fuzzy_v1_*",
                "--out", str(EXTRACT), "--with-c4", "--zip"],
               cwd=REPO, check=True)
ANALYSIS = Path("/kaggle/working/fuzzy_v1_analysis")
analysis = subprocess.run([sys.executable, "-m", "src.analysis.fuzzy_comparison",
                "--runs-root", str(RUNS), "--expected-configs",
                str(REPO / "configs/fuzzy_v1"), "--out", str(ANALYSIS),
                "--plan", str(REPO / "configs/analysis_plan.json")], cwd=REPO)
print("ANALYSIS_EXIT", analysis.returncode, flush=True)
shutil.make_archive("/kaggle/working/fuzzy_v1_analysis", "zip", ANALYSIS)
shutil.make_archive("/kaggle/working/fuzzy_v1_full_runs", "zip", RUNS)
print("FUZZY_MAIN_COMPLETE", flush=True)